# Regional Momentum Factor Analysis

Download **total market return** (`Mkt = Mkt-RF + RF`) and **momentum** (`WML`) from the [Ken French Data Library](https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/data_library.html) for:

- North America
- Europe
- Asia ex Japan
- Japan

Each regional dataset is stored in a pandas DataFrame indexed by **month-end date**.

In [ ]:
# Auto-reload edited modules so changes in src/ take effect without a kernel restart.
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

# Make the project root importable so `from src...` works from the notebooks/ folder.
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src" / "ken_french.py").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.ken_french import REGIONS, load_all_regions

# Cache the downloaded CSVs here (gitignored).
DATA_DIR = PROJECT_ROOT / "data" / "ken_french"
DATA_DIR.mkdir(parents=True, exist_ok=True)

# Show floats with 2 decimals for readable tables.
pd.set_option("display.float_format", lambda x: f"{x:8.2f}")

## Download and load regional factors

For each region we merge two Ken French CSV files:

| File | Key columns |
|------|-------------|
| Regional 3-Factor | `Mkt-RF`, `SMB`, `HML`, `RF` |
| Regional Momentum | `WML` (Winners Minus Losers) |

We compute **total market return** as `Mkt = Mkt-RF + RF`.

In [ ]:
# Download + parse all four regions in one call.
regional_factors = load_all_regions()

# Save a copy of each region and print a quick summary.
for region_key, df in regional_factors.items():
    csv_path = DATA_DIR / f"{region_key}_factors.csv"
    df.to_csv(csv_path)
    label = REGIONS[region_key]["label"]
    print(f"{label:<16} rows={len(df):>4}  {df.index.min().date()} to {df.index.max().date()}")

## Inspect DataFrames (date-indexed)

In [ ]:
# Convenient named handles for each region's DataFrame.
north_america = regional_factors["north_america"]
europe = regional_factors["europe"]
asia_ex_japan = regional_factors["asia_ex_japan"]
japan = regional_factors["japan"]

north_america.head()

In [ ]:
europe.tail()

In [ ]:
# Sanity check: confirm the date index and columns look right and nothing is missing.
for region_key, df in regional_factors.items():
    print(f"\n{REGIONS[region_key]['label']}")
    print(f"  index name: {df.index.name}")
    print(f"  index dtype: {df.index.dtype}")
    print(f"  columns: {list(df.columns)}")
    print(f"  missing values: {int(df.isna().sum().sum())}")

## Single combined DataFrame

One date-indexed DataFrame with **one column per region/factor pair**:
`{Region}_Mkt` (total market return) and `{Region}_WML` (momentum).

In [ ]:
# Pull the market (Mkt) and momentum (WML) column from each region into one wide table.
panel_series = {}
for region_key, meta in REGIONS.items():
    region_df = regional_factors[region_key]
    panel_series[f"{meta['label']}_Mkt"] = region_df["Mkt"]
    panel_series[f"{meta['label']}_WML"] = region_df["WML"]

factors_df = pd.DataFrame(panel_series).sort_index()
factors_df.index.name = "date"

factors_df.to_csv(DATA_DIR / "regional_mkt_wml_panel.csv")
factors_df.head()

## Monthly returns table

Long-format table (one row per date + region) with:

| Column | Meaning |
|--------|---------|
| `monthly_return` | Total market return (`Mkt`), % |
| `momentum_return` | Momentum return (`WML`), % |
| `roll36m_ann_return` | Rolling 36-month **geometric annualized** market return, % |
| `roll36m_ann_momentum` | Rolling 36-month **geometric annualized** momentum return, % |
| `market_drawdown` | Drawdown of the market equity curve vs its running peak, % (recovers to 0) |
| `momentum_drawdown` | Drawdown of the momentum equity curve vs its running peak, % (recovers to 0) |

In [ ]:
WINDOW = 36  # rolling window length in months


def rolling_annualized(returns_pct: pd.Series, window: int = WINDOW) -> pd.Series:
    """Geometric annualized return over a rolling window, in %.

    Compound the window's monthly returns, then scale to a yearly rate:
    (product of (1 + r))^(12 / window) - 1.
    """
    growth = 1 + returns_pct / 100
    window_growth = growth.rolling(window).apply(lambda x: x.prod(), raw=True)
    return (window_growth ** (12 / window) - 1) * 100


def drawdown(returns_pct: pd.Series) -> pd.Series:
    """Drawdown of the compounded equity curve vs its running peak, in %."""
    equity = (1 + returns_pct / 100).cumprod()
    return (equity / equity.cummax() - 1) * 100


# Build the long (tidy) table region by region so rolling windows stay in time order.
monthly_parts = []
for region_key, meta in REGIONS.items():
    region_df = regional_factors[region_key]
    market = region_df["Mkt"]
    momentum = region_df["WML"]

    monthly_parts.append(
        pd.DataFrame(
            {
                "region": meta["label"],
                "monthly_return": market,
                "momentum_return": momentum,
                "roll36m_ann_return": rolling_annualized(market),
                "roll36m_ann_momentum": rolling_annualized(momentum),
                "market_drawdown": drawdown(market),
                "momentum_drawdown": drawdown(momentum),
            }
        )
    )

monthly_returns_table = (
    pd.concat(monthly_parts)
    .reset_index(names="date")
    .sort_values(["date", "region"])
    .reset_index(drop=True)
)

monthly_returns_table.to_csv(DATA_DIR / "monthly_returns_long.csv", index=False)
# First 35 months have no 36m rolling value; show the first fully-populated rows.
monthly_returns_table.dropna().head(12)

In [ ]:
monthly_returns_table.tail()

## Momentum factor statistics (WML)

WML is a **self-financing long/short** portfolio, so its return is already an excess return — no risk-free rate adjustment.

- **Ann. Mean (%)** = mean(WML) × 12  
- **Ann. Vol (%)** = std(WML) × √12  
- **Sharpe** = Ann. Mean / Ann. Vol  
- **t-stat** = mean(WML) / (std(WML) / √n)  
- **Max DD (%)** and **Date** = deepest drawdown of the market and momentum equity curves, and when it occurred

In [54]:
rows = []
for region_key, meta in REGIONS.items():
    region_df = regional_factors[region_key]
    wml = region_df["WML"]  # monthly momentum returns, in %
    market = region_df["Mkt"]

    monthly_mean = wml.mean()
    monthly_vol = wml.std()
    n_obs = wml.count()

    # Annualize: mean scales by 12, volatility by sqrt(12) (returns assumed independent).
    ann_mean = monthly_mean * 12
    ann_vol = monthly_vol * (12**0.5)

    # WML is long/short (self-financing), so it is already an excess return -> RF = 0.
    sharpe = ann_mean / ann_vol
    # t-stat on the monthly mean: is the average return reliably different from zero?
    t_stat = monthly_mean / (monthly_vol / (n_obs**0.5))

    # Max drawdown = the deepest point of each equity curve, plus when it occurred.
    market_dd = drawdown(market)
    momentum_dd = drawdown(wml)

    rows.append(
        {
            "Region": meta["label"],
            "Ann. Mean (%)": ann_mean,
            "Ann. Vol (%)": ann_vol,
            "Sharpe": sharpe,
            "t-stat": t_stat,
            "Market Max DD (%)": market_dd.min(),
            "Market Max DD Date": market_dd.idxmin().date(),
            "Momentum Max DD (%)": momentum_dd.min(),
            "Momentum Max DD Date": momentum_dd.idxmin().date(),
        }
    )

momentum_table = pd.DataFrame(rows).set_index("Region")
momentum_table

,Ann. Mean (%),Ann. Vol (%),Sharpe,t-stat,Market Max DD (%),Market Max DD Date,Momentum Max DD (%),Momentum Max DD Date
Region,,,,,,,,
North America,6.63,15.71,0.42,2.52,-51.07,2009-02-28,-49.97,2009-09-30
Europe,10.58,13.13,0.81,4.80,-58.97,2009-02-28,-45.59,2009-09-30
Asia ex Japan,10.48,14.31,0.73,4.37,-60.05,2009-02-28,-52.81,2000-11-30
Japan,1.43,14.48,0.10,0.59,-58.10,2003-03-31,-41.96,2012-02-29


## Cumulative performance (compounded)

Four panels:
1. **Momentum (WML)** — growth of $1 (log scale)
2. **Total market (Mkt)** — growth of $1 (log scale)
3. **Rolling 36-month annualized market return** — a rate, linear % axis
4. **Rolling 36-month annualized momentum return** — a rate, linear % axis

Monthly returns compound geometrically, so growth is `(1 + r).cumprod()`, not a running sum.

In [ ]:
# Returns are in %, so divide by 100 and compound: growth of $1 = product of (1 + r).
growth = (1 + factors_df / 100).cumprod()

mkt_cols = [col for col in factors_df.columns if col.endswith("_Mkt")]
wml_cols = [col for col in factors_df.columns if col.endswith("_WML")]

# Rolling 36-month annualized returns (a rate, so linear % axis — not growth).
roll36m_return = factors_df[mkt_cols].apply(rolling_annualized)
roll36m_momentum = factors_df[wml_cols].apply(rolling_annualized)

# Fixed color per region so lines and their max-drawdown markers stay matched.
region_labels = [REGIONS[key]["label"] for key in REGIONS]
region_color = dict(zip(region_labels, plt.cm.tab10.colors))

# Date of the deepest drawdown per region, for the vertical markers.
market_dd_date = {REGIONS[k]["label"]: drawdown(regional_factors[k]["Mkt"]).idxmin() for k in REGIONS}
momentum_dd_date = {REGIONS[k]["label"]: drawdown(regional_factors[k]["WML"]).idxmin() for k in REGIONS}


def plot_panel(ax, data, suffix, title, ylabel, dd_dates, log_scale=False):
    for column in data.columns:
        region = column.replace(suffix, "")
        ax.plot(data.index, data[column], label=region, color=region_color[region])
        # Dashed vertical line at this region's max-drawdown date, same color.
        ax.axvline(dd_dates[region], color=region_color[region], linestyle="--", alpha=0.5)
    ax.set_title(title)
    ax.set_ylabel(ylabel)
    if log_scale:
        ax.set_yscale("log")  # keeps multi-decade compounding readable
    else:
        ax.axhline(0, color="black", linewidth=0.8)
    ax.legend(loc="upper left")
    ax.grid(alpha=0.3)


fig, axes = plt.subplots(4, 1, figsize=(12, 16), sharex=True)
plot_panel(axes[0], growth[wml_cols], "_WML", "Momentum (WML) — Growth of $1 (compounded)", "Growth of $1", momentum_dd_date, log_scale=True)
plot_panel(axes[1], growth[mkt_cols], "_Mkt", "Total Market (Mkt) — Growth of $1 (compounded)", "Growth of $1", market_dd_date, log_scale=True)
plot_panel(axes[2], roll36m_return, "_Mkt", "Market — Rolling 36-Month Annualized Return", "Annualized return (%)", market_dd_date)
plot_panel(axes[3], roll36m_momentum, "_WML", "Momentum — Rolling 36-Month Annualized Return", "Annualized return (%)", momentum_dd_date)

plt.tight_layout()
plt.show()

## Drawdowns

Depth below the running peak (`drawdown`). It sits at 0 during new highs and dips negative during declines — the "underwater" view of each equity curve.

In [ ]:
# Drawdown (depth below running peak) for each region's market and momentum curves.
market_dd = factors_df[mkt_cols].apply(drawdown)
momentum_dd = factors_df[wml_cols].apply(drawdown)


def plot_dd(ax, data, suffix, title):
    for column in data.columns:
        region = column.replace(suffix, "")
        ax.plot(data.index, data[column], label=region, color=region_color[region])
    ax.set_title(title)
    ax.set_ylabel("Drawdown (%)")
    ax.legend(loc="lower left")
    ax.grid(alpha=0.3)


fig, axes = plt.subplots(2, 1, figsize=(12, 9), sharex=True)
plot_dd(axes[0], market_dd, "_Mkt", "Market — Drawdown")
plot_dd(axes[1], momentum_dd, "_WML", "Momentum — Drawdown")

plt.tight_layout()
plt.show()

## Findings 

Europe and Apac x japan shows great systematic return for momentum, with t-stat >>2
NA t-stat >2, Momentum factor less performing than Europe and APAC xJapan
Japan t-stat < 1 --> cannot differentiate from 0 premium, but we also see Japan market value stagnent for some time, is this a structural deleveraging issue, or momentum is really "broken" here?

2009 is pretty clear, 

Momentum not so affected by Covid